<a href="https://colab.research.google.com/github/julurisaichandu/nlp/blob/main/notebook_07_hmm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notebook 7: Hidden Markov Models
===============

CS 6120 Natural Language Processing, Amir



Saichandu Juluri

Saving notebooks as pdfs
----------

Feel free to add cells to this notebook as you wish. Make sure to leave **code that you've written** and any **answers to questions** that you've written in your notebook. Turn in your notebook as a pdf at the end of lecture's day.


To convert your notebook to a pdf for turn in, you'll do the following:
1. Kernel -> Restart & Run All (clear your kernel's memory and run all cells)
2. File -> Download As -> .html -> open in a browser -> print to pdf

(The download as pdf option doesn't preserve formatting and output as nicely as taking the step "through" html, but will do if the above doesn't work for you.)


**Credit**
*Viterbi implementation was adapted from a notebook by Felix Muzny*

Task 1: Implement a Hidden Markov Model for POS tagging
-------

In this notebook you will implement a Hidden Markov Model for Part-of-Speech Tagging of Twitter data. We will use the dataset proposed by [Gimpel et al. (2011)](http://www.cs.cmu.edu/~ark/TweetNLP/gimpel+etal.acl11.pdf) which defines a label set consisting of 25 tags (see Table 1 of the paper for a description of the tags).

Recall that HMMs are generative probabilistic sequence models that model $P(x_1^N, y_1^N)$ the joint probability of a sequences observations ${x} \in \mathcal{V}$ and tags ${y} \in \mathcal{\tau}$. Here, observations are words from a vocabulary $\mathcal{V}$. The model uses a markov assumption to predict a sequence of tags given a sequence of observations as


$\argmax_{y_1, \ldots, y_N}P(y_1, \ldots, y_N| x_1, \ldots, x_N )= P(y_1) P(x_1|y_1) P(y_2|y_1) P(x_2|y_2) \ldots P(y_{\text{STOP}}|y_N)$

The model is parametrized by:
-  $\mathbf{\pi} \in \mathbb{R}^{\tau}$, vector encoding a probability distribution over initial tags $P(y_1)$
- $\mathbf{E} \in \mathbb{R}^{\mathcal{\tau} \times \mathcal{V}}$, a matrix of emission probabilities $P(x|y)$
- $\mathbf{T} \in \mathbb{R}^{\mathcal{\tau} \times \mathcal{\tau}}$, a square matrix of transition probabilities $P(y_{\text{curr}}|y_{\text{prev}})$

The parameters can be estimated with Maximum Likelihood Estimation via counting and normalizing (like other generative models that we have seen before).

$P(y_1 = k) = \frac{\text{count}(y_1 = k)}{\sum_{k' \in \mathcal{\tau}}\text{count}(y_1 = k')}$

$P(x|y) = \frac{\text{count}(x_i,y_i)}{\text{count}(y_i)}$


$P(y_{\text{curr}}|y_{\text{prev}}) = \frac{\text{count}(y_{\text{prev}}, y_{\text{curr}})}{\text{count}(y_{\text{prev}})}$

Inference in this type of model is tricky due to the exponential number of possible assignments. The viterbi algorithm uses dynamic programming to efficiently perform inference.


In [3]:
from collections import defaultdict, Counter

In [4]:
def read_data(path):
    tweet_pos_data = []
    with open(path) as f:
        curr = []
        for l in f.readlines():
            x = l.split()
            # print(x)
            if len(x) == 2:
                curr.append(tuple(x))
            elif len(x) == 0:
                tweet_pos_data.append(curr)
                curr = []
    return tweet_pos_data

def get_x(data):
    all_xs = []
    for d in data:
        xs = [x[0] for x in d]
        all_xs.append(xs)
    return all_xs


In [24]:
class HMM:

    def __init__(self):
        #parameters
        self.pi = Counter()
        self.T = defaultdict(Counter)
        self.E = defaultdict(Counter)
        self.Y = set()

    def train(self, data: list):

        self.hmm_count(data)
        self.hmm_normalize()


    def hmm_count(self, data: list):
        # iterate through sentences and tokens/labels
        for sentence in data:
            # (word, tag)

            # Adding all tags to the set of possible tags
            for word, tag in sentence:
                self.Y.add(tag)

            # storing counts of initial states
            if sentence:
                first_word, first_tag = sentence[0]
                self.pi[first_tag] += 1

            # storing counts of emissions
            for word, tag in sentence:
                self.E[tag][word] += 1

            # storing counts of transitions
            for i in range(len(sentence)-1):
                curr_word, curr_tag = sentence[i]
                next_word, next_tag = sentence[i+1]
                self.T[curr_tag][next_tag] += 1

    def hmm_normalize(self):
      norm_emms = []
      norm_trans = []
      norm_initial = 0
      # normalize initial states
      total_initial = sum(self.pi.values())
      if total_initial > 0:  # Avoid division by zero
          for tag in self.pi:
              self.pi[tag] /= total_initial
              # testing intials
              norm_initial +=self.pi[tag]
      print("norm initial distribution\n",norm_initial)
      # normalize emissions
      for tag in self.E:
          total_emissions = sum(self.E[tag].values())
          if total_emissions > 0:  # Avoid division by zero
              for word in self.E[tag]:
                  self.E[tag][word] /= total_emissions
          # testing emissions
          norm_emms.append(sum(self.E[tag].values()))
      print("norm emmision distribution\n",norm_emms)

      # normalize transitions
      for prev_tag in self.T:
          total_transitions = sum(self.T[prev_tag].values())
          if total_transitions > 0:  # Avoid division by zero
              for curr_tag in self.T[prev_tag]:
                  self.T[prev_tag][curr_tag] /= total_transitions
           # testing emissions
          norm_trans.append(sum(self.T[prev_tag].values()))
      print("norm transition distribution\n",norm_trans)

    def viterbi(self, o: list) -> (dict, dict):
        """
        Run the viterbi algorithm for a bi-gram based HMM
        params:
        o - a list of tokens (observations)

        return:
        probability_table - a list of dictionaries of states to probabilities,
        one dictionary per word in the test data that represents the
        probability of being at that state for that word
        pointer_table - a list of dictionaries of states to states,
        one dictionary per word in the test data that represents the
        backpointers for which previous state led to the best probability
        for the current state
        """

        prev_prob = 0

        # initialize the probability and backpointer tables
        # so that we have one dictionary per column/token in the
        # sentence
        viterbi_table = [{} for i in range(len(o))]
        backpointers = [{} for i in range(len(o))]

        for t in range(len(o)):
            # current observation/word
            o_t = o[t]
            # consider every tag
            for y in self.Y:
                # P(x_i | y_i)
                emission = self.E[y][o_t]
                # initial probabilities (base case)
                if t == 0:
                    prob = self.pi[y] * emission
                    viterbi_table[t][y] = prob
                    backpointers[t][y] = None
                else:
                    # consider all possible transitions from previous step (recursion)
                    max_prob = 0
                    best_y = None
                    for prev_y in self.Y:
                        # P(y_i|y_i - 1)
                        transition = self.T[prev_y][y]
                        # previous score
                        prev_prob = viterbi_table[t - 1][prev_y]
                        # candidate probability
                        candidate_prob = prev_prob * transition * emission
                        # keep track of best score/tag thus far
                        if candidate_prob > max_prob or best_y is None:
                            max_prob = candidate_prob
                            best_y = prev_y
                    # fill in the probability/pointer tables for this observation/state combo
                    viterbi_table[t][y] = max_prob
                    backpointers[t][y] = best_y

        return viterbi_table, backpointers

    def greedy_decoding(self, o: list) -> (dict, dict):
        """
        Greedily decode the sequence of tags to go along with a sequence of words

        params:
        o - a list of tokens (observations)
        return:
        max_seq - a tuple of the best sequence of tags for the observation
        max_prob - a float of the final calculated probability for that sequence of tags
        """
        greedy_seq = []
        for t in range(len(o)):
            # current observation/word
            o_t = o[t]

            probabilities = {}
            # consider every state
            for state in self.Y:
                # P(w_i | t_i)
                # TODO: get the emission probability for a word
                emission = self.E[state][o_t]
                # see if we're in the first column
                if t == 0:
                    prob = self.pi[state] * emission
                    probabilities[state] = prob
                else:
                    # consider ONLY THE BEST previous state
                    # as the state that we came from
                    # (this is what makes this strategy different than Viterbi!)
                    # TODO: get the appropriate transition probability
                    transition = self.T[greedy_seq[-1]][state]

                    # calculate the new probability
                    # TODO FILL THIS IN
                    prob = transition * emission
                    probabilities[state] = prob

            # choose the best state for the prev state
            max_prob = max(probabilities.values())
            # get the argmax
            max_state = max(probabilities, key=probabilities.get, default=())
            # build up our sequence
            greedy_seq.append(max_state)

        return greedy_seq, max_prob

    def recover_labels(self, o: list, probability_table: dict, pointer_table: dict) -> list:
        """
        Recovers the set of labels from a given probability and pointer table and matches
        them to corresponding elements in a list of observations.

        params:
        o - a list of tokens
        probability_table - a list of dictionaries of states to probabilities,
        one dictionary per word in the test data that represents the
        probability of being at that state for that word
        pointer_table - a list of dictionaries of states to states,
        one dictionary per word in the test data that represents the
        backpointers for which previous state led to the best probability
        for the current state
        return:
        a list of tuples in the format [(token, label), (token, label)...]
        """
        # start in the final column
        labels = []
        max_final = None
        max_final_state = None
        final_col = probability_table[len(probability_table) - 1]
        for state in final_col:
            if max_final is None or final_col[state] > max_final:
                max_final = final_col[state]
                max_final_state = state

        labels = [(o[-1], max_final_state)]
        curr = max_final_state
        for i in range(len(o) - 1, 0, -1):
            label = pointer_table[i][curr]
            labels.insert(0, (o[i - 1], label))
            curr = label
        return labels

    def inference(self, o: list):

        viterbi_table, backpointers = self.viterbi(o)
        labels = self.recover_labels(o, viterbi_table, backpointers)
        return labels


In [11]:
train_data = read_data("oct27.tsv")
test_x = get_x(train_data)
print(test_x[10])
print(train_data[1])

['Just', 'watched', 'the', 'Jersey', 'Shore', 'episode', 'of', 'Southpark', '.', 'Really', "don't", 'like', 'South', 'park', ',', 'but', 'that', 'was', 'hilarious', '!!!!!!!']
[('RT', '~'), ('@DjBlack_Pearl', '@'), (':', '~'), ('wat', 'O'), ('muhfuckaz', 'N'), ('wearin', 'V'), ('4', 'P'), ('the', 'D'), ('lingerie', 'N'), ('party', 'N'), ('?????', ',')]


Q1: Implement the methods `hmm_count()` and `hmm_normalize`. Note that you need to iterate through training examples and for each example iterate through the tokens/labels. Convince yourself that the parameters were correctly estimated by testing that the emission probabilities P(w|y) form a valid distribution (i.e, confirm that for a given y, all values are between 0-1 and they sum to 1). Perform the same test for the initial and transition probabilities.

In [25]:
hmm = HMM()
hmm.train(train_data)
example = "I loved the movie :)".split()
hmm.inference(example)

norm initial distribution
 0.9999999999999999
norm emmision distribution
 [1.0000000000000036, 0.9999999999999832, 1.000000000000001, 0.999999999999986, 0.9999999999999807, 1.0000000000000004, 0.9999999999999996, 1.0000000000000162, 0.9999999999999991, 0.9999999999999994, 0.9999999999999912, 0.9999999999999999, 1.0000000000000024, 1.0000000000000004, 1.0000000000000004, 1.0, 0.9999999999999969, 1.0000000000000022, 1.000000000000002, 1.0, 1.0, 0.9999999999999998, 0.9999999999999996, 0.9999999999999999, 1.0]
O
O
O
O
O
O
O
O
O
O
O
O
O
O
O
O
O
O
O
O
O
V
V
V
V
V
V
V
V
V
V
V
V
V
V
V
V
V
V
V
V
V
V
V
D
D
D
D
D
D
D
D
D
D
D
D
D
D
D
D
A
A
A
A
A
A
A
A
A
A
A
A
A
A
A
A
A
A
A
A
N
N
N
N
N
N
N
N
N
N
N
N
N
N
N
N
N
N
N
N
N
N
P
P
P
P
P
P
P
P
P
P
P
P
P
P
P
P
P
P
P
P
P
P
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
^
^
^
^
^
^
^
^
^
^
^
^
^
^
^
^
^
^
^
^
^
^
L
L
L
L
L
L
L
L
L
L
L
L
L
L
~
~
~
~
~
~
~
~
~
~
~
~
~
~
~
~
~
~
~
~
~
@
@
@
@
@
@
@
@
@
@
@
@
@
@
@
@
@
@
@
@
$
$
$
$
$
$
$
$
$
$
$
$
$
$
$
$
$
$
!
!


[('I', 'O'), ('loved', 'V'), ('the', 'D'), ('movie', 'N'), (':)', 'E')]

Q2: Experiment with your trained HMM by running a few examples through the model. We have already implemented the viterbi algorithm for you

In [18]:
i=0
for i in range(10):
    print(hmm.inference(test_x[i]))
    print()

[('I', 'O'), ('predict', 'V'), ('I', 'O'), ("won't", 'V'), ('win', 'V'), ('a', 'D'), ('single', 'A'), ('game', 'N'), ('I', 'O'), ('bet', 'V'), ('on', 'T'), ('.', ','), ('Got', 'V'), ('Cliff', '^'), ('Lee', '^'), ('today', 'N'), (',', ','), ('so', 'R'), ('if', 'P'), ('he', 'O'), ('loses', 'V'), ('its', 'L'), ('on', 'P'), ('me', 'O'), ('RT', '~'), ('@e_one', '@'), (':', '~'), ('Texas', '^'), ('(', ','), ('cont', '~'), (')', ','), ('http://tl.gd/6meogh', 'U')]

[('RT', '~'), ('@DjBlack_Pearl', '@'), (':', '~'), ('wat', 'O'), ('muhfuckaz', 'N'), ('wearin', 'V'), ('4', 'P'), ('the', 'D'), ('lingerie', 'N'), ('party', 'N'), ('?????', ',')]

[('Wednesday', '^'), ('27th', '$'), ('october', '^'), ('2010', '$'), ('.', ','), ('》have', 'V'), ('a', 'D'), ('nice', 'A'), ('day', 'N'), (':)', 'E')]

[('RT', '~'), ('@ddlovato', '@'), (':', '~'), ('@joejonas', '@'), ('oh', '!'), (',', ','), ('hey', '!'), ('THANKS', 'N'), ('jerk', 'N'), ('!', ',')]

[('@thecamion', '@'), ('I', 'O'), ('like', 'V'), ('monk

OPTIONAL Q3: Implement greedy decoding and compare the predictions against viterbi decoding.